O pré-processamento típico segue essa ordem:

- Olhar os tipos (df.info()) e ver o que devia ser número/data mas veio como object/str

- Olhar valores únicos (df["COLUNA"].unique()) pra achar sujeira: espaços, formatos inconsistentes.

- Limpar (strip, trocar separador, etc.)

- Converter pro tipo certo

- Só então calcular estatísticas, agrupar, plotar

In [64]:
import pandas as pd

df = pd.read_csv("../dados/brutos/ufc_fight_stats.csv")

In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42012 entries, 0 to 42011
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EVENT       42012 non-null  str    
 1   BOUT        42012 non-null  str    
 2   ROUND       41970 non-null  str    
 3   FIGHTER     41970 non-null  str    
 4   KD          41970 non-null  float64
 5   SIG.STR.    41970 non-null  str    
 6   SIG.STR. %  41970 non-null  str    
 7   TOTAL STR.  41970 non-null  str    
 8   TD          41970 non-null  str    
 9   TD %        41970 non-null  str    
 10  SUB.ATT     41970 non-null  float64
 11  REV.        41970 non-null  float64
 12  CTRL        41970 non-null  str    
 13  HEAD        41970 non-null  str    
 14  BODY        41970 non-null  str    
 15  LEG         41970 non-null  str    
 16  DISTANCE    41970 non-null  str    
 17  CLINCH      41970 non-null  str    
 18  GROUND      41970 non-null  str    
dtypes: float64(3), str(16)
memory usage:

In [66]:
resultados = pd.read_csv("../dados/processados/ufc_fight_results.csv")
chave = ["EVENT", "BOUT"]

# Garante que as chaves estão no mesmo formato nas duas tabelas
df["EVENT"] = df["EVENT"].str.strip()
df["BOUT"] = df["BOUT"].str.strip()

# Confere se a chave é única na tabela de referência (senão o merge duplica linhas)
print("Chave única em resultados?", not resultados.duplicated(chave).any())

antes = len(df)
df = df.merge(resultados[chave], on=chave, how="inner", validate="many_to_one")
print(f"Linhas: {antes} -> {len(df)}")


Chave única em resultados? True
Linhas: 42012 -> 41040


In [67]:
# Conferência: todas as lutas da referência têm estatísticas?
lutas_stats = df[chave].drop_duplicates()
sem_stats = resultados[chave].merge(lutas_stats, on=chave, how="left", indicator=True)
print(sem_stats["_merge"].value_counts())   # o ideal é tudo em "both"


_merge
both          8625
left_only        0
right_only       0
Name: count, dtype: int64


In [68]:
var = 'TOTAL STR.'

df[var].unique()
# df[var].count()

<StringArray>
[ '49 of 98',  '44 of 85',  '49 of 84',  '15 of 23',  '22 of 37',  '32 of 60',
  '62 of 94',  '36 of 57',  '63 of 80',   '5 of 12',
 ...
  '62 of 66',  '51 of 57', '90 of 103',  '54 of 58',  '73 of 77',  '40 of 41',
  '84 of 94',   '1 of 36',  '64 of 99',  '56 of 57']
Length: 4632, dtype: str

In [69]:
df[var]

0        49 of 98
1        44 of 85
2        49 of 84
3        15 of 23
4        22 of 37
           ...   
41035    22 of 36
41036    20 of 39
41037     7 of 13
41038    12 of 23
41039      0 of 4
Name: TOTAL STR., Length: 41040, dtype: str

Excluir os nulos e tranformar em Int


In [70]:
df = df.dropna(subset=["FIGHTER"])          # linhas sem lutador = linha vazia
df['KD'] = df['KD'].astype(int)


Uma coluna somente com o número de golpes significativos

In [71]:
df["SIG.STR."] = df["SIG.STR."].str.split(" of ").str[0].astype(int)

Trocando um texto de '10%' para 0.10

In [72]:
df["SIG.STR. %"] = pd.to_numeric(df["SIG.STR. %"].str.rstrip("%"), errors="coerce") / 100
df["TD %"]       = pd.to_numeric(df["TD %"].str.rstrip("%"), errors="coerce") / 100



Uma coluna somente com o número de golpes

In [73]:


partes = df["TOTAL STR."].str.split(" of ", expand=True).astype(int)

df["TOTAL STR. %"] = partes[0] / partes[1]   # 0 tentativas -> divisão por zero -> NaN
df["TOTAL STR."]   = partes[0]               # sobrescreve só no fim
